[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/101.%20The%20Supply%20Chain%20Finance%20%26%20Working%20Capital%20Optimization%20Problem/P101-Tier-7_executed.ipynb)

---


# Problem 101, Tier 7: Human–AI Partnership for Supply Chain Finance

## Purpose and scope

This notebook demonstrates a rule-based simulation of collaboration between AI recommendations and human experts in supply chain finance (SCF). It represents an interaction process: the system produces an explainable recommendation, a human expert accepts or modifies it, a simulated outcome is recorded, and trust measures are updated. The later sections summarize and visualize the simulated interactions.

**Inputs** are expert profiles, decision scenarios, and illustrative context values such as cash flow, inventory turnover, financing need, and risk score. **Outputs** are recommendations, human-review decisions, simulated outcomes, trust and collaboration metrics, and visual summaries. The code uses Python rules and random draws; it does not train a machine-learning model, estimate causal effects, or solve a financial optimization problem.

### Model assumptions

- Recommendation rules are demonstrations, not validated financial advice. Context values and outcome values are illustrative and should not be interpreted as forecasts or realized savings.
- Human review is simulated probabilistically from recommendation confidence, trust, decision style, and whether the expert's listed expertise matches the decision type. This is not a behavioral or psychological model of real experts.
- Decision outcomes are random draws from simple distributions. They are useful for illustrating the interaction and metric pipeline, but they do not establish that accepting or modifying a recommendation caused an outcome.
- Trust updates use bounded heuristic rules. Integrity, benevolence, competence, predictability, and overall trust are descriptive scores, not empirically calibrated quantities. Benevolence is initialized but not updated by the current rules.
- The ROI section uses stated illustrative cost and benefit assumptions. Those estimates are not an audited business case; a payback period cannot be inferred without specifying a time horizon and the timing of costs and benefits.

The notebook proceeds in this order: define the data structures, implement the partnership rules, create experts, run collaborative sessions, analyze trust and outcomes, visualize the results, and calculate illustrative analytics. Random outcomes and review decisions may differ between runs because no random seed is specified.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

print("Required packages imported successfully.")

## Human–AI partnership data structures

The data classes below give each part of the interaction a clear role. An `AIRecommendation` stores the proposed action, its confidence, rationale, supporting context, and alternatives. A `HumanExpert` stores an expert's profile and accumulated interaction history. A `PartnershipInteraction` records one review and its simulated outcome. `TrustMetrics` stores the trust dimensions for one expert, while `SymbioticPartnership` groups the records for the whole collaboration.

A confidence score is represented on a scale from 0 to 1. In this simulation, it is a heuristic score assigned by the recommendation rules; it is not a statistically calibrated probability that an action will succeed.

In [ ]:
@dataclass
class AIRecommendation:
    """An AI proposal together with its explanation and context."""

    recommendation_id: str
    decision_type: str
    recommended_action: str
    confidence: float
    reasoning: List[str]
    supporting_data: Dict[str, Any]
    alternatives: List[Tuple[str, float]]
    timestamp: datetime = field(default_factory=datetime.now)
    ethical_considerations: List[str] = field(default_factory=list)


@dataclass
class HumanExpert:
    """A human expert profile and its interaction history."""

    expert_id: str
    name: str
    expertise_areas: List[str]
    experience_years: int
    decision_style: str
    trust_score: float = 0.5
    interaction_history: List[str] = field(default_factory=list)
    expertise_confidence: Dict[str, float] = field(default_factory=dict)


@dataclass
class PartnershipInteraction:
    """A record of one human review and its simulated outcome."""

    interaction_id: str
    expert_id: str
    recommendation_id: str
    ai_confidence: float
    human_acceptance: bool
    human_modification: Optional[str]
    final_decision: str
    decision_outcome: float
    interaction_time: datetime = field(default_factory=datetime.now)
    satisfaction_score: float = 0.0


@dataclass
class TrustMetrics:
    """Heuristic trust scores for one expert–AI relationship."""

    expert_id: str
    overall_trust: float
    competence_trust: float
    integrity_trust: float
    benevolence_trust: float
    predictability_trust: float
    last_updated: datetime = field(default_factory=datetime.now)


@dataclass
class SymbioticPartnership:
    """Stored experts, recommendations, interactions, and partnership metrics."""

    partnership_id: str
    experts: Dict[str, HumanExpert] = field(default_factory=dict)
    ai_recommendations: Dict[str, AIRecommendation] = field(default_factory=dict)
    interactions: Dict[str, PartnershipInteraction] = field(default_factory=dict)
    trust_metrics: Dict[str, TrustMetrics] = field(default_factory=dict)
    partnership_age: int = 0
    total_interactions: int = 0
    agreement_rate: float = 0.0


print("Human–AI partnership data structures defined.")

## Symbiotic partnership implementation

The implementation follows the interaction sequence used throughout the notebook:

1. Apply decision-type-specific rules to the scenario context and create an explainable recommendation.
2. Calculate a human acceptance probability from confidence, trust, decision style, and expertise match; draw a simulated review decision.
3. Generate a simulated outcome, record the interaction, and update the expert's trust metrics.
4. Aggregate recorded interactions into partnership metrics.

For an expert \(e\) and recommendation \(r\), the review probability begins as \(p_{e,r}=c_r t_e\), where \(c_r\) is recommendation confidence and \(t_e\) is overall trust. The code then applies style and expertise multipliers and caps the probability at 0.95. A random draw determines acceptance. Thus the review is a stochastic simulation, not a deterministic optimization or an inference about real human behavior.

For a recorded interaction, the outcome is sampled from a normal distribution: accepted recommendations use the first alternative's illustrative value as the mean and 5,000 as the standard deviation; modified recommendations use a mean of 0 and a standard deviation of 2,000. These assumptions make the example runnable, but do not measure financial performance. Trust changes are heuristic increments or decrements, bounded to the interval [0.1, 0.95].

In [ ]:
class HumanAISymbioticPartnership:
    """Rule-based human–AI collaboration simulation for supply chain finance."""

    def __init__(self, partnership_id: str):
        self.partnership = SymbioticPartnership(partnership_id=partnership_id)
        self.decision_context = {
            "market_conditions": "stable",
            "risk_appetite": "moderate",
            "strategic_priorities": ["growth", "efficiency"],
        }

    def add_expert(self, expert: HumanExpert) -> None:
        """Register an expert and initialize that expert's trust scores."""
        self.partnership.experts[expert.expert_id] = expert
        self.partnership.trust_metrics[expert.expert_id] = TrustMetrics(
            expert_id=expert.expert_id,
            overall_trust=0.5,
            competence_trust=0.5,
            integrity_trust=0.7,
            benevolence_trust=0.6,
            predictability_trust=0.5,
        )

    def generate_ai_recommendation(
        self, decision_type: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Generate and store a recommendation using the relevant rule set."""
        recommendation_id = (
            f"AI-REC-{len(self.partnership.ai_recommendations) + 1:03d}"
        )
        generators = {
            "payment_terms": self._generate_payment_terms_recommendation,
            "working_capital": self._generate_working_capital_recommendation,
            "financing": self._generate_financing_recommendation,
            "risk": self._generate_risk_recommendation,
        }
        if decision_type in generators:
            recommendation = generators[decision_type](recommendation_id, context)
        else:
            recommendation = self._generate_general_recommendation(
                recommendation_id, decision_type, context
            )
        self.partnership.ai_recommendations[recommendation_id] = recommendation
        return recommendation

    def _generate_payment_terms_recommendation(
        self, rec_id: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Apply cash-flow and supplier-relationship rules to payment terms."""
        current_terms = context.get("current_terms", 45)
        cash_flow = context.get("cash_flow", 100000)
        supplier_relationship = context.get("supplier_relationship", "good")
        reasoning = []
        confidence = 0.75

        if cash_flow > 150000:
            recommended_action = "Extend payment terms by 15 days"
            reasoning.append(
                f"Strong cash flow (${cash_flow:,.0f}) supports longer payment terms"
            )
            confidence += 0.1
        elif cash_flow < 50000:
            recommended_action = "Reduce payment terms by 10 days"
            reasoning.append(
                f"Limited cash flow (${cash_flow:,.0f}) requires shorter terms"
            )
            confidence += 0.05
        else:
            recommended_action = "Maintain current payment terms"
            reasoning.append(
                f"Moderate cash flow (${cash_flow:,.0f}) supports current terms"
            )
            confidence += 0.15

        if supplier_relationship == "excellent":
            reasoning.append("A strong supplier relationship may support negotiation")
            confidence += 0.05
        elif supplier_relationship == "poor":
            reasoning.append("A weak supplier relationship may limit term changes")
            confidence -= 0.1
            recommended_action = "Maintain current payment terms"

        confidence = max(0.3, min(0.95, confidence))
        return AIRecommendation(
            recommendation_id=rec_id,
            decision_type="payment_terms",
            recommended_action=recommended_action,
            confidence=confidence,
            reasoning=reasoning,
            supporting_data={"cash_flow": cash_flow, "current_terms": current_terms},
            alternatives=[
                ("Extend by 10 days", cash_flow * 0.02),
                ("Reduce by 5 days", cash_flow * -0.01),
                ("Maintain status quo", 0),
            ],
            ethical_considerations=[
                "Consider fair treatment of suppliers",
                "Consider supply chain stability",
            ],
        )

    def _generate_working_capital_recommendation(
        self, rec_id: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Apply inventory-turnover and receivables rules to working capital."""
        current_wc = context.get("working_capital", 200000)
        inventory_turnover = context.get("inventory_turnover", 6)
        receivables_days = context.get("receivables_days", 45)
        reasoning = []
        confidence = 0.7

        if inventory_turnover < 4:
            recommended_action = "Implement inventory reduction program"
            reasoning.append(
                f"Low inventory turnover ({inventory_turnover:.1f}x) may indicate excess inventory"
            )
            confidence += 0.1
        elif receivables_days > 60:
            recommended_action = "Accelerate receivables collection"
            reasoning.append(
                f"High receivables days ({receivables_days:.0f}) delay cash conversion"
            )
            confidence += 0.1
        else:
            recommended_action = "Optimize working capital cycle"
            reasoning.append("Current metrics may allow further optimization")
            confidence += 0.05

        confidence = max(0.3, min(0.95, confidence))
        return AIRecommendation(
            recommendation_id=rec_id,
            decision_type="working_capital",
            recommended_action=recommended_action,
            confidence=confidence,
            reasoning=reasoning,
            supporting_data={
                "working_capital": current_wc,
                "inventory_turnover": inventory_turnover,
                "receivables_days": receivables_days,
            },
            alternatives=[
                ("Dynamic discounting", current_wc * 0.03),
                ("Supply chain financing", current_wc * 0.04),
                ("Status quo", 0),
            ],
        )

    def _generate_financing_recommendation(
        self, rec_id: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Apply credit-score and urgency rules to financing decisions."""
        financing_need = context.get("financing_need", 100000)
        credit_score = context.get("credit_score", 750)
        urgency = context.get("urgency", "medium")
        reasoning = []
        confidence = 0.8

        if credit_score > 800 and urgency == "low":
            recommended_action = "Secure traditional bank financing"
            reasoning.append(
                f"A high credit score ({credit_score}) may qualify for favorable rates"
            )
            confidence += 0.1
        elif urgency == "high":
            recommended_action = "Utilize supply chain financing"
            reasoning.append("High urgency favors faster financing options")
            confidence += 0.05
        else:
            recommended_action = "Consider mixed financing approach"
            reasoning.append("A mixed approach may balance cost and speed")
            confidence += 0.05

        confidence = max(0.3, min(0.95, confidence))
        return AIRecommendation(
            recommendation_id=rec_id,
            decision_type="financing",
            recommended_action=recommended_action,
            confidence=confidence,
            reasoning=reasoning,
            supporting_data={
                "financing_need": financing_need,
                "credit_score": credit_score,
                "urgency": urgency,
            },
            alternatives=[
                ("Invoice discounting", financing_need * 0.08),
                ("Asset-based lending", financing_need * 0.10),
                ("Equity financing", financing_need * 0.15),
            ],
        )

    def _generate_risk_recommendation(
        self, rec_id: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Apply risk-score and concentration rules to risk management."""
        risk_score = context.get("risk_score", 0.5)
        concentration_risk = context.get("concentration_risk", 0.3)
        market_volatility = context.get("market_volatility", "medium")
        reasoning = []
        confidence = 0.75

        if risk_score > 0.7:
            recommended_action = "Implement comprehensive risk mitigation"
            reasoning.append(
                f"High risk score ({risk_score:.2f}) calls for prompt attention"
            )
            confidence += 0.1
        elif concentration_risk > 0.5:
            recommended_action = "Diversify supplier and customer base"
            reasoning.append(
                f"High concentration risk ({concentration_risk:.2f}) suggests diversification"
            )
            confidence += 0.05
        else:
            recommended_action = "Maintain current risk monitoring"
            reasoning.append("The supplied risk indicators support continued monitoring")
            confidence += 0.1

        confidence = max(0.3, min(0.95, confidence))
        return AIRecommendation(
            recommendation_id=rec_id,
            decision_type="risk",
            recommended_action=recommended_action,
            confidence=confidence,
            reasoning=reasoning,
            supporting_data={
                "risk_score": risk_score,
                "concentration_risk": concentration_risk,
                "market_volatility": market_volatility,
            },
            alternatives=[
                ("Insurance coverage", -risk_score * 50000),
                ("Hedging strategy", -risk_score * 30000),
                ("Self-insurance", -risk_score * 20000),
            ],
        )

    def _generate_general_recommendation(
        self, rec_id: str, decision_type: str, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Provide a generic recommendation for an unlisted decision type."""
        return AIRecommendation(
            recommendation_id=rec_id,
            decision_type=decision_type,
            recommended_action=f"Optimize {decision_type} based on current conditions",
            confidence=0.6,
            reasoning=["Use the available context to consider a general optimization."],
            supporting_data=dict(context),
            alternatives=[
                ("Status quo", 0),
                ("Moderate change", 1000),
                ("Aggressive change", 5000),
            ],
        )

    def human_review_decision(
        self, expert_id: str, recommendation: AIRecommendation
    ) -> Tuple[bool, Optional[str], float]:
        """Simulate acceptance, modification, and satisfaction for one review."""
        if expert_id not in self.partnership.experts:
            raise KeyError(f"Unknown expert ID: {expert_id}")
        if expert_id not in self.partnership.trust_metrics:
            raise KeyError(f"Trust metrics are missing for expert ID: {expert_id}")

        expert = self.partnership.experts[expert_id]
        trust = self.partnership.trust_metrics[expert_id]
        acceptance_probability = recommendation.confidence * trust.overall_trust

        if expert.decision_style == "conservative":
            acceptance_probability *= 0.8
        elif expert.decision_style == "aggressive":
            acceptance_probability *= 1.2

        if recommendation.decision_type in expert.expertise_areas:
            acceptance_probability *= 1.1

        acceptance_probability = max(0.0, min(0.95, acceptance_probability))
        accepts = random.random() < acceptance_probability

        if accepts:
            modification = None
            satisfaction = 0.8 + random.uniform(-0.1, 0.2)
        else:
            modifications = {
                "conservative": "Reduce scope and implement gradually",
                "aggressive": "Accelerate implementation and expand scope",
                "balanced": "Adjust timing and add monitoring",
            }
            modification = modifications.get(
                expert.decision_style, "Adjust the recommendation and monitor results"
            )
            satisfaction = 0.6

        return accepts, modification, satisfaction

    def record_interaction(
        self,
        expert_id: str,
        recommendation: AIRecommendation,
        accepts: bool,
        modification: Optional[str],
        satisfaction: float,
    ) -> PartnershipInteraction:
        """Record a review, simulate its outcome, and update partnership state."""
        if expert_id not in self.partnership.experts:
            raise KeyError(f"Unknown expert ID: {expert_id}")
        if recommendation.recommendation_id not in self.partnership.ai_recommendations:
            raise ValueError("The recommendation must be generated by this partnership.")
        if not 0.0 <= satisfaction <= 1.0:
            raise ValueError("Satisfaction must be between 0 and 1.")
        if not accepts and modification is None:
            raise ValueError("A modified decision must include the human modification.")
        if not recommendation.alternatives:
            raise ValueError("At least one alternative is required to simulate an outcome.")

        interaction_id = f"INT-{len(self.partnership.interactions) + 1:03d}"
        if accepts:
            outcome = random.gauss(recommendation.alternatives[0][1], 5000)
            final_decision = recommendation.recommended_action
        else:
            outcome = random.gauss(0, 2000)
            final_decision = modification

        interaction = PartnershipInteraction(
            interaction_id=interaction_id,
            expert_id=expert_id,
            recommendation_id=recommendation.recommendation_id,
            ai_confidence=recommendation.confidence,
            human_acceptance=accepts,
            human_modification=modification,
            final_decision=final_decision,
            decision_outcome=outcome,
            satisfaction_score=satisfaction,
        )
        self.partnership.interactions[interaction_id] = interaction
        self.partnership.total_interactions += 1
        self.partnership.experts[expert_id].interaction_history.append(interaction_id)
        self._update_trust_metrics(expert_id, interaction)
        interaction_count = len(self.partnership.interactions)
        accepted_count = sum(
            item.human_acceptance
            for item in self.partnership.interactions.values()
        )
        self.partnership.agreement_rate = accepted_count / interaction_count
        return interaction

    def _update_trust_metrics(
        self, expert_id: str, interaction: PartnershipInteraction
    ) -> None:
        """Apply bounded heuristic trust changes from the interaction record."""
        trust = self.partnership.trust_metrics[expert_id]

        if interaction.human_acceptance and interaction.decision_outcome > 0:
            trust.competence_trust += 0.05
            trust.overall_trust += 0.03
        elif not interaction.human_acceptance and interaction.decision_outcome < 0:
            trust.overall_trust += 0.02
        elif interaction.human_acceptance and interaction.decision_outcome < 0:
            trust.competence_trust -= 0.03
            trust.overall_trust -= 0.02

        if interaction.satisfaction_score > 0.8:
            trust.integrity_trust += 0.02
        elif interaction.satisfaction_score < 0.5:
            trust.integrity_trust -= 0.01

        confidence_alignment = abs(
            interaction.ai_confidence - float(interaction.human_acceptance)
        )
        trust.predictability_trust += (1 - confidence_alignment) * 0.02

        for attribute in (
            "competence_trust",
            "integrity_trust",
            "predictability_trust",
            "overall_trust",
        ):
            current_value = getattr(trust, attribute)
            setattr(trust, attribute, max(0.1, min(0.95, current_value)))
        trust.last_updated = datetime.now()
        self.partnership.experts[expert_id].trust_score = trust.overall_trust

    def get_partnership_metrics(self) -> Dict[str, Any]:
        """Aggregate interaction, trust, satisfaction, and outcome measures."""
        interactions = list(self.partnership.interactions.values())
        if not interactions:
            trust_values = [
                trust.overall_trust
                for trust in self.partnership.trust_metrics.values()
            ]
            return {
                "total_interactions": 0,
                "agreement_rate": 0.0,
                "average_trust": float(np.mean(trust_values)) if trust_values else 0.5,
                "average_satisfaction": 0.0,
                "total_outcome": 0.0,
                "average_outcome_per_interaction": 0.0,
            }

        total_outcome = sum(item.decision_outcome for item in interactions)
        trust_values = [
            trust.overall_trust
            for trust in self.partnership.trust_metrics.values()
        ]
        return {
            "total_interactions": len(interactions),
            "agreement_rate": np.mean(
                [item.human_acceptance for item in interactions]
            ),
            "average_trust": float(np.mean(trust_values)) if trust_values else 0.5,
            "average_satisfaction": float(
                np.mean([item.satisfaction_score for item in interactions])
            ),
            "total_outcome": total_outcome,
            "average_outcome_per_interaction": total_outcome / len(interactions),
        }


print("Human–AI partnership rules implemented.")

## Partnership setup and expert profiles

Four experts are registered with different primary expertise areas, experience levels, and decision styles. The expertise profile determines whether an expert receives the expertise-match multiplier during review. Trust scores are initialized by `add_expert`; the listed expertise-confidence values are profile information and do not directly alter the current review rule.

In [ ]:
partnership = HumanAISymbioticPartnership("SCF-HAI-001")

experts_config = [
    (
        "EXP-001",
        "Dr. Sarah Chen",
        "finance",
        15,
        "balanced",
        {"finance": 0.9, "risk": 0.8, "supply_chain": 0.7, "strategy": 0.8},
    ),
    (
        "EXP-002",
        "Mark Rodriguez",
        "supply_chain",
        12,
        "conservative",
        {"supply_chain": 0.9, "finance": 0.7, "risk": 0.8, "strategy": 0.6},
    ),
    (
        "EXP-003",
        "Prof. James Liu",
        "risk",
        20,
        "conservative",
        {"risk": 0.95, "finance": 0.8, "supply_chain": 0.6, "strategy": 0.7},
    ),
    (
        "EXP-004",
        "Amanda Foster",
        "strategy",
        8,
        "aggressive",
        {"strategy": 0.85, "finance": 0.7, "supply_chain": 0.8, "risk": 0.6},
    ),
]

all_expertise_areas = ["finance", "supply_chain", "risk", "strategy"]
for expert_id, name, primary_expertise, years, style, expertise_confidence in experts_config:
    expertise_areas = [primary_expertise] + [
        area
        for area in all_expertise_areas
        if area != primary_expertise and expertise_confidence.get(area, 0) > 0.6
    ]
    partnership.add_expert(
        HumanExpert(
            expert_id=expert_id,
            name=name,
            expertise_areas=expertise_areas,
            experience_years=years,
            decision_style=style,
            expertise_confidence=expertise_confidence,
        )
    )

print("Partnership setup complete.")
print(f"Partnership ID: {partnership.partnership.partnership_id}")
print(f"Number of human experts: {len(partnership.partnership.experts)}")

for expert_id, expert in partnership.partnership.experts.items():
    print(f"\n{expert.name} ({expert_id})")
    print(f"  Primary expertise: {expert.expertise_areas[0]}")
    print(f"  Experience: {expert.experience_years} years")
    print(f"  Decision style: {expert.decision_style}")
    print(
        "  Initial trust: "
        f"{partnership.partnership.trust_metrics[expert_id].overall_trust:.2f}"
    )
    print(f"  Expertise confidence: {expert.expertise_confidence}")

decision_styles = {
    expert.decision_style for expert in partnership.partnership.experts.values()
}
covered_areas = {
    area
    for expert in partnership.partnership.experts.values()
    for area in expert.expertise_areas
}
average_experience = np.mean(
    [expert.experience_years for expert in partnership.partnership.experts.values()]
)
initial_average_trust = np.mean(
    [trust.overall_trust for trust in partnership.partnership.trust_metrics.values()]
)

print("\nPartnership configuration:")
print(f"Decision context: {partnership.decision_context}")
print(f"Distinct decision styles: {sorted(decision_styles)}")
print(f"Expertise coverage: {len(covered_areas)}/{len(all_expertise_areas)} areas")
print(f"Average experience: {average_experience:.1f} years")
print(f"Initial average trust: {initial_average_trust:.3f}")

## Collaborative decision-making sessions

Each scenario pairs a decision type and context with a primary expert. The notebook creates a recommendation, displays its explanation, simulates the expert's review, records the interaction, and reports the updated trust score. Since the review and outcome use random draws, individual session results vary across runs. The summary values computed by the following cells describe only the interactions generated in the current run.

In [ ]:
scenarios = [
    {
        "name": "Supplier Payment Terms Negotiation",
        "decision_type": "payment_terms",
        "context": {
            "current_terms": 45,
            "cash_flow": 180000,
            "supplier_relationship": "excellent",
            "market_conditions": "favorable",
        },
        "primary_expert": "EXP-001",
    },
    {
        "name": "Working Capital Optimization",
        "decision_type": "working_capital",
        "context": {
            "working_capital": 350000,
            "inventory_turnover": 3.2,
            "receivables_days": 68,
            "payables_days": 42,
        },
        "primary_expert": "EXP-002",
    },
    {
        "name": "Supply Chain Risk Assessment",
        "decision_type": "risk",
        "context": {
            "risk_score": 0.72,
            "concentration_risk": 0.58,
            "market_volatility": "high",
            "geopolitical_risk": "medium",
        },
        "primary_expert": "EXP-003",
    },
    {
        "name": "Growth Financing Strategy",
        "decision_type": "financing",
        "context": {
            "financing_need": 250000,
            "credit_score": 780,
            "urgency": "medium",
            "growth_projections": 0.15,
        },
        "primary_expert": "EXP-004",
    },
]

session_results = []
print("Collaborative decision-making sessions\n")

for session_number, scenario in enumerate(scenarios, start=1):
    expert_id = scenario["primary_expert"]
    expert = partnership.partnership.experts[expert_id]
    print(f"Session {session_number}: {scenario['name']}")
    print(f"Primary expert: {expert.name}")

    recommendation = partnership.generate_ai_recommendation(
        scenario["decision_type"], scenario["context"]
    )
    print("\nAI recommendation:")
    print(f"  Action: {recommendation.recommended_action}")
    print(f"  Heuristic confidence: {recommendation.confidence:.2f}")
    print("  Rationale:")
    for reason in recommendation.reasoning:
        print(f"    - {reason}")
    if recommendation.ethical_considerations:
        print("  Ethical considerations:")
        for consideration in recommendation.ethical_considerations:
            print(f"    - {consideration}")

    accepts, modification, satisfaction = partnership.human_review_decision(
        expert_id, recommendation
    )
    print(f"\nExpert review ({expert.name}):")
    print(f"  Decision: {'Accept' if accepts else 'Modify'}")
    if modification is not None:
        print(f"  Modification: {modification}")
    final_action = recommendation.recommended_action if accepts else modification
    print(f"  Final action: {final_action}")
    print(f"  Satisfaction score: {satisfaction:.2f}")

    interaction = partnership.record_interaction(
        expert_id,
        recommendation,
        accepts,
        modification,
        satisfaction,
    )
    trust = partnership.partnership.trust_metrics[expert_id]
    print(f"  Simulated outcome: ${interaction.decision_outcome:,.0f}")
    print(f"  Updated overall trust: {trust.overall_trust:.3f}")

    session_results.append(
        {
            "session": session_number,
            "scenario": scenario["name"],
            "expert": expert.name,
            "ai_confidence": recommendation.confidence,
            "human_acceptance": accepts,
            "satisfaction": satisfaction,
            "outcome": interaction.decision_outcome,
            "trust": trust.overall_trust,
        }
    )
    print("\n" + "=" * 72 + "\n")

metrics = partnership.get_partnership_metrics()
print("Session summary:")
print("Session | Scenario | Expert | AI confidence | Accepted | Satisfaction | Outcome")
for result in session_results:
    print(
        f"{result['session']:>7} | {result['scenario']} | {result['expert']} | "
        f"{result['ai_confidence']:.2f} | {result['human_acceptance']} | "
        f"{result['satisfaction']:.2f} | ${result['outcome']:,.0f}"
    )

print("\nPartnership metrics for this run:")
print(f"Total interactions: {metrics['total_interactions']}")
print(f"Agreement rate: {metrics['agreement_rate']:.1%}")
print(f"Average trust: {metrics['average_trust']:.3f}")
print(f"Average satisfaction: {metrics['average_satisfaction']:.3f}")
print(f"Total simulated outcome: ${metrics['total_outcome']:,.0f}")
print(
    "Average simulated outcome per interaction: "
    f"${metrics['average_outcome_per_interaction']:,.0f}"
)

## Trust development and relationship analysis

Trust metrics are reported separately because the simulation updates different dimensions using different rules. The performance table groups recorded interactions by expert. Decision-style comparisons are descriptive only: each style has few observations in this demonstration, so they cannot support statistical conclusions. Expertise impact is grouped by the recommendation's decision type; it is not a causal estimate of expertise.

In [ ]:
print("Trust development and relationship analysis\n")

trust_analysis = []
print(
    "Expert | Overall | Competence | Integrity | Benevolence | Predictability"
)
for expert_id, trust in partnership.partnership.trust_metrics.items():
    expert = partnership.partnership.experts[expert_id]
    trust_change = trust.overall_trust - 0.5
    print(
        f"{expert.name} | {trust.overall_trust:.3f} | "
        f"{trust.competence_trust:.3f} | {trust.integrity_trust:.3f} | "
        f"{trust.benevolence_trust:.3f} | {trust.predictability_trust:.3f}"
    )
    trust_analysis.append(
        {
            "expert": expert.name,
            "style": expert.decision_style,
            "overall_trust": trust.overall_trust,
            "competence_trust": trust.competence_trust,
            "integrity_trust": trust.integrity_trust,
            "benevolence_trust": trust.benevolence_trust,
            "predictability_trust": trust.predictability_trust,
            "trust_change": trust_change,
        }
    )

expert_performance = {}
for expert_id, expert in partnership.partnership.experts.items():
    expert_interactions = [
        interaction
        for interaction in partnership.partnership.interactions.values()
        if interaction.expert_id == expert_id
    ]
    if expert_interactions:
        total_outcome = sum(
            interaction.decision_outcome for interaction in expert_interactions
        )
        expert_performance[expert_id] = {
            "name": expert.name,
            "interactions": len(expert_interactions),
            "acceptance_rate": np.mean(
                [item.human_acceptance for item in expert_interactions]
            ),
            "average_satisfaction": np.mean(
                [item.satisfaction_score for item in expert_interactions]
            ),
            "total_outcome": total_outcome,
            "average_outcome": total_outcome / len(expert_interactions),
        }

print("\nExpert performance (descriptive):")
for performance in expert_performance.values():
    print(
        f"{performance['name']}: "
        f"{performance['interactions']} interaction(s), "
        f"acceptance {performance['acceptance_rate']:.1%}, "
        f"satisfaction {performance['average_satisfaction']:.2f}, "
        f"total simulated outcome ${performance['total_outcome']:,.0f}, "
        f"average ${performance['average_outcome']:,.0f}"
    )

style_analysis = {}
for expert_id, expert in partnership.partnership.experts.items():
    entry = style_analysis.setdefault(
        expert.decision_style,
        {"trust": [], "acceptance_rates": []},
    )
    entry["trust"].append(
        partnership.partnership.trust_metrics[expert_id].overall_trust
    )
    if expert_id in expert_performance:
        entry["acceptance_rates"].append(
            expert_performance[expert_id]["acceptance_rate"]
        )

print("\nDecision-style summaries (small descriptive sample):")
for style, values in style_analysis.items():
    average_acceptance = (
        float(np.mean(values["acceptance_rates"]))
        if values["acceptance_rates"]
        else float("nan")
    )
    print(
        f"{style.capitalize()}: {len(values['trust'])} expert(s), "
        f"average trust {np.mean(values['trust']):.3f}, "
        f"average acceptance {average_acceptance:.1%}"
    )

expertise_impact = {}
for interaction in partnership.partnership.interactions.values():
    recommendation = partnership.partnership.ai_recommendations[
        interaction.recommendation_id
    ]
    decision_type = recommendation.decision_type
    entry = expertise_impact.setdefault(
        decision_type,
        {"outcomes": [], "acceptances": [], "satisfaction": []},
    )
    entry["outcomes"].append(interaction.decision_outcome)
    entry["acceptances"].append(interaction.human_acceptance)
    entry["satisfaction"].append(interaction.satisfaction_score)

print("\nSummary by recommendation type:")
for decision_type, values in expertise_impact.items():
    print(
        f"{decision_type}: average simulated outcome "
        f"${np.mean(values['outcomes']):,.0f}, "
        f"acceptance {np.mean(values['acceptances']):.1%}, "
        f"average satisfaction {np.mean(values['satisfaction']):.2f}"
    )

maturity_factors = {
    "Trust development": 20 if metrics["average_trust"] > 0.7 else 10,
    "High satisfaction": 20 if metrics["average_satisfaction"] > 0.8 else 10,
    "Positive average outcome": (
        20 if metrics["average_outcome_per_interaction"] > 5000 else 10
    ),
    "High agreement rate": 20 if metrics["agreement_rate"] > 0.7 else 10,
    "Repeated interactions": 20 if metrics["total_interactions"] >= 10 else 10,
}
maturity_score = sum(maturity_factors.values())
print("\nIllustrative partnership maturity checklist:")
for factor, score in maturity_factors.items():
    print(f"{factor}: {score}/20")
print(f"Checklist score: {maturity_score}/100")

if maturity_score >= 90:
    maturity_level = "High maturity by this checklist"
elif maturity_score >= 70:
    maturity_level = "Established collaboration by this checklist"
elif maturity_score >= 50:
    maturity_level = "Developing collaboration by this checklist"
else:
    maturity_level = "Early-stage collaboration by this checklist"
print(f"Checklist category: {maturity_level}")

## Human–AI partnership visualization

The four panels summarize trust dimensions, expert-level review measures, simulated outcomes, and results grouped by decision type. Bars and points describe this run only. In the expert comparison, outcome values are scaled by the largest absolute average outcome so that positive and negative values remain interpretable on a shared percentage-like display; the underlying currency amounts remain available in the analytics above.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Human–AI Partnership in Supply Chain Finance",
    fontsize=16,
    fontweight="bold",
)

# Trust dimensions show the current scores for each expert.
expert_names = [item["expert"] for item in trust_analysis]
trust_dimensions = [
    "overall_trust",
    "competence_trust",
    "integrity_trust",
    "benevolence_trust",
    "predictability_trust",
]
dimension_labels = [
    "Overall",
    "Competence",
    "Integrity",
    "Benevolence",
    "Predictability",
]
colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#592E83"]
x_positions = np.arange(len(expert_names))
bar_width = 0.15
for index, (dimension, color) in enumerate(zip(trust_dimensions, colors)):
    values = [item[dimension] for item in trust_analysis]
    axes[0, 0].bar(
        x_positions + index * bar_width,
        values,
        bar_width,
        label=dimension_labels[index],
        color=color,
        alpha=0.85,
    )
axes[0, 0].set_title("Trust scores by dimension")
axes[0, 0].set_ylabel("Heuristic trust score (0–1)")
axes[0, 0].set_xticks(x_positions + 2 * bar_width)
axes[0, 0].set_xticklabels(expert_names, rotation=35, ha="right")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].legend(fontsize=8)

# Expert outcomes are scaled by their largest absolute average value.
performance_items = list(expert_performance.values())
performance_names = [item["name"] for item in performance_items]
acceptance_rates = [item["acceptance_rate"] * 100 for item in performance_items]
satisfaction_scores = [
    item["average_satisfaction"] * 100 for item in performance_items
]
average_outcomes = [item["average_outcome"] for item in performance_items]
outcome_scale = max([abs(value) for value in average_outcomes], default=0.0)
normalized_outcomes = [
    value / outcome_scale * 100 if outcome_scale else 0.0
    for value in average_outcomes
]
x_performance = np.arange(len(performance_names))
comparison_width = 0.25
axes[0, 1].bar(
    x_performance - comparison_width,
    acceptance_rates,
    comparison_width,
    label="Acceptance rate (%)",
    color="green",
    alpha=0.7,
)
axes[0, 1].bar(
    x_performance,
    satisfaction_scores,
    comparison_width,
    label="Satisfaction score (%)",
    color="blue",
    alpha=0.7,
)
axes[0, 1].bar(
    x_performance + comparison_width,
    normalized_outcomes,
    comparison_width,
    label="Scaled average outcome (%)",
    color="orange",
    alpha=0.7,
)
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set_title("Descriptive expert comparison")
axes[0, 1].set_ylabel("Percentage or scaled outcome")
axes[0, 1].set_xticks(x_performance)
axes[0, 1].set_xticklabels(performance_names, rotation=35, ha="right")
axes[0, 1].legend(fontsize=8)

# Point color identifies acceptance; point size represents satisfaction.
interactions = list(partnership.partnership.interactions.values())
session_numbers = np.arange(1, len(interactions) + 1)
outcomes = [item.decision_outcome for item in interactions]
point_colors = [
    "green" if item.human_acceptance else "red" for item in interactions
]
point_sizes = [100 + 200 * item.satisfaction_score for item in interactions]
axes[1, 0].scatter(
    session_numbers,
    outcomes,
    c=point_colors,
    s=point_sizes,
    alpha=0.75,
    edgecolors="black",
)
axes[1, 0].axhline(0, color="black", linestyle="--", alpha=0.5)
axes[1, 0].set_title("Simulated outcome by session")
axes[1, 0].set_xlabel("Session number")
axes[1, 0].set_ylabel("Simulated outcome (USD; illustrative)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(
    handles=[
        plt.Line2D([], [], marker="o", linestyle="", color="green", label="Accepted"),
        plt.Line2D([], [], marker="o", linestyle="", color="red", label="Modified"),
    ],
    loc="best",
)

# Decision types use a shared currency axis for outcomes and a percentage axis for acceptance.
decision_types = list(expertise_impact)
average_outcomes_by_type = [
    float(np.mean(expertise_impact[item]["outcomes"])) for item in decision_types
]
acceptance_by_type = [
    float(np.mean(expertise_impact[item]["acceptances"])) * 100
    for item in decision_types
]
x_types = np.arange(len(decision_types))
type_width = 0.35
axes[1, 1].bar(
    x_types - type_width / 2,
    average_outcomes_by_type,
    type_width,
    label="Average simulated outcome (USD)",
    color="purple",
    alpha=0.7,
)
axes[1, 1].set_title("Summary by decision type")
axes[1, 1].set_ylabel("Average simulated outcome (USD; illustrative)")
axes[1, 1].set_xticks(x_types)
axes[1, 1].set_xticklabels(
    [item.replace("_", " ").title() for item in decision_types],
    rotation=30,
    ha="right",
)
axes[1, 1].axhline(0, color="black", linewidth=0.8)
acceptance_axis = axes[1, 1].twinx()
acceptance_axis.bar(
    x_types + type_width / 2,
    acceptance_by_type,
    type_width,
    label="Acceptance rate (%)",
    color="brown",
    alpha=0.55,
)
acceptance_axis.set_ylabel("Acceptance rate (%)")
acceptance_axis.set_ylim(0, 100)
outcome_handles, outcome_labels = axes[1, 1].get_legend_handles_labels()
acceptance_handles, acceptance_labels = acceptance_axis.get_legend_handles_labels()
axes[1, 1].legend(
    outcome_handles + acceptance_handles,
    outcome_labels + acceptance_labels,
    fontsize=8,
    loc="best",
)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Advanced partnership analytics

These indicators combine interaction records with explicit illustrative assumptions. They are descriptive diagnostics, not validated measures of organizational performance.

- **Decision alignment** counts whether high-confidence recommendations were accepted and lower-confidence recommendations were not. The cutoff is 0.7; the score is a simple proportion, not a calibration statistic.
- **Expertise utilization** measures the fraction of interactions in which the recommendation type appears in the expert's listed expertise areas.
- **Trust change** is the mean change from the initial overall-trust score of 0.5 across experts; it is not a time-normalized velocity.
- **Satisfaction trend** compares the later portion of the ordered interaction list with the earlier portion. A short sequence makes this especially uncertain.
- **Outcome consistency** is reported as a bounded descriptive score based on the observed outcome spread and mean. It should not be interpreted as a forecast of future variability.

The ROI arithmetic uses the stated cost and benefit assumptions in the code. The current scenario outcomes are not annual benefits, and one-time and recurring effects are not separated. Consequently, the notebook reports an illustrative ROI percentage but does not label a cost-to-benefit ratio as a payback period.

In [ ]:
print("Advanced partnership analytics\n")

interactions = list(partnership.partnership.interactions.values())
collaboration_metrics = {
    "decision_alignment": 0.0,
    "expertise_utilization": 0.0,
    "mean_trust_change": 0.0,
    "satisfaction_trend": 0.0,
    "outcome_consistency": 0.0,
}

if interactions:
    alignment_count = sum(
        (item.ai_confidence > 0.7 and item.human_acceptance)
        or (item.ai_confidence <= 0.7 and not item.human_acceptance)
        for item in interactions
    )
    collaboration_metrics["decision_alignment"] = alignment_count / len(interactions)

    expertise_matches = 0
    for interaction in interactions:
        recommendation = partnership.partnership.ai_recommendations[
            interaction.recommendation_id
        ]
        expert = partnership.partnership.experts[interaction.expert_id]
        expertise_matches += recommendation.decision_type in expert.expertise_areas
    collaboration_metrics["expertise_utilization"] = (
        expertise_matches / len(interactions)
    )

trust_changes = [item["trust_change"] for item in trust_analysis]
if trust_changes:
    collaboration_metrics["mean_trust_change"] = float(np.mean(trust_changes))

satisfaction_series = [item.satisfaction_score for item in interactions]
if len(satisfaction_series) > 1:
    split_index = len(satisfaction_series) // 2
    first_part = satisfaction_series[:split_index]
    later_part = satisfaction_series[split_index:]
    collaboration_metrics["satisfaction_trend"] = float(
        np.mean(later_part) - np.mean(first_part)
    )

outcome_series = [item.decision_outcome for item in interactions]
if len(outcome_series) > 1:
    outcome_mean = float(np.mean(outcome_series))
    outcome_std = float(np.std(outcome_series))
    collaboration_metrics["outcome_consistency"] = (
        1 - outcome_std / (outcome_std + abs(outcome_mean))
        if outcome_std + abs(outcome_mean) > 0
        else 1.0
    )

print("Collaboration indicators:")
for name, value in collaboration_metrics.items():
    print(f"{name.replace('_', ' ').title()}: {value:.3f}")

decision_patterns = {}
for expert_id, expert in partnership.partnership.experts.items():
    expert_interactions = [
        item for item in interactions if item.expert_id == expert_id
    ]
    if not expert_interactions:
        continue
    patterns = {
        "accepts_high_confidence": 0,
        "rejects_low_confidence": 0,
        "modifies_moderate_confidence": 0,
        "total_decisions": len(expert_interactions),
    }
    for interaction in expert_interactions:
        if interaction.ai_confidence > 0.8 and interaction.human_acceptance:
            patterns["accepts_high_confidence"] += 1
        elif interaction.ai_confidence < 0.5 and not interaction.human_acceptance:
            patterns["rejects_low_confidence"] += 1
        elif 0.5 <= interaction.ai_confidence <= 0.8 and not interaction.human_acceptance:
            patterns["modifies_moderate_confidence"] += 1
    decision_patterns[expert_id] = patterns

print("\nDecision patterns by expert:")
for expert_id, patterns in decision_patterns.items():
    expert = partnership.partnership.experts[expert_id]
    print(
        f"{expert.name}: accepts high confidence="
        f"{patterns['accepts_high_confidence']}, rejects low confidence="
        f"{patterns['rejects_low_confidence']}, modifies moderate confidence="
        f"{patterns['modifies_moderate_confidence']}, "
        f"total={patterns['total_decisions']}"
    )

# These monetary inputs are illustrative assumptions, not measured costs.
expert_cost_per_hour = 150
average_time_per_decision_hours = 0.5
ai_development_cost = 200000
training_cost_per_expert = 5000
expert_count = len(partnership.partnership.experts)
interaction_count = metrics["total_interactions"]

total_expert_cost = (
    expert_count
    * expert_cost_per_hour
    * average_time_per_decision_hours
    * interaction_count
)
total_training_cost = expert_count * training_cost_per_expert
total_cost = ai_development_cost + total_training_cost + total_expert_cost

# The recorded outcomes are treated as an illustrative value component only.
decision_value = metrics["total_outcome"]
efficiency_gain = interaction_count * 2000
risk_reduction = total_cost * 0.15
knowledge_capture = expert_count * 10000
total_benefits = (
    decision_value + efficiency_gain + risk_reduction + knowledge_capture
)
partnership_roi = (total_benefits - total_cost) / total_cost * 100

print("\nIllustrative ROI calculation (assumption-based):")
print(f"AI development cost: ${ai_development_cost:,.2f}")
print(f"Training cost: ${total_training_cost:,.2f}")
print(f"Expert time cost: ${total_expert_cost:,.2f}")
print(f"Total modeled cost: ${total_cost:,.2f}")
print(f"Recorded outcome component: ${decision_value:,.2f}")
print(f"Assumed efficiency gains: ${efficiency_gain:,.2f}")
print(f"Assumed risk reduction: ${risk_reduction:,.2f}")
print(f"Assumed knowledge capture: ${knowledge_capture:,.2f}")
print(f"Total modeled benefits: ${total_benefits:,.2f}")
print(f"Illustrative ROI: {partnership_roi:.1f}%")
print("Payback period: not estimated; benefit timing and time horizon are unspecified.")

potential_factors = {
    "scalability": min(1.0, expert_count / 10),
    "learning_capability": metrics["average_trust"],
    "decision_quality": min(
        1.0,
        max(0.0, metrics["average_outcome_per_interaction"] / 10000),
    ),
    "expertise_transfer": collaboration_metrics["expertise_utilization"],
    "automation_potential": metrics["agreement_rate"],
}
potential_score = float(np.mean(list(potential_factors.values()))) * 100

print("\nIllustrative future-potential indicators:")
for factor, score in potential_factors.items():
    print(f"{factor.replace('_', ' ').title()}: {score:.1%}")
print(f"Mean indicator score: {potential_score:.1f}/100")
print(
    "Interpretation: these heuristic indicators summarize this simulation; "
    "they are not validated predictions of future performance."
)

## Summary and conclusions

This notebook demonstrates a human–AI collaboration workflow for supply chain finance. It connects explainable, rule-based recommendations with simulated human review, interaction recording, heuristic trust updates, descriptive analytics, and visual summaries.

### How to interpret the results

- The recommendation rules show how decision context can be translated into an action and a textual rationale. Their confidence scores are heuristic values, not calibrated probabilities.
- Acceptance, satisfaction, and outcomes are generated by simulation rules and random draws. A result from one run is not evidence that a particular action is financially beneficial or that one decision style is superior.
- Trust, maturity, potential, and ROI values depend on explicit rules and illustrative assumptions. They should not be used as production thresholds or investment evidence.
- The scenarios cover payment terms, working capital, financing, and risk. The code is a compact teaching example; it does not represent all contractual, regulatory, ethical, or operational constraints in real SCF decisions.

### Strengths and limitations of the approach

The workflow makes recommendation rationale and human intervention visible, keeps records that can be analyzed, and illustrates how trust heuristics can be connected to interactions. Human review can support accountability in consequential decisions, but it also requires appropriate expertise and clear governance. The simulation does not establish that a human–AI partnership outperforms a human-only or AI-only process; demonstrating that claim would require a suitable comparison design and evidence.

The main limitations are the simplified recommendation rules, small illustrative scenario set, stochastic outcomes, heuristic trust updates, and assumption-based financial indicators. Real deployment would require validated data, domain-specific controls, independent evaluation, monitoring, privacy and security safeguards, and human accountability appropriate to the decision.

The intended lesson is the structure of the collaboration process—not a financial recommendation or a claim of production readiness.